# Comparison Metrics for Different SNID Analyses

This notebook compares the performance of SNID across different datasets: CfA, CSP, Modjaz (Fixed Type), Modjaz (Fixed Subtype), and ZTF.

Metrics calculated in 5-day bins of true age (Limited to -15 to 25 days):
- Bias (Mean Residual)
- Spread (Standard Deviation of Residuals)
- RMSE
- Count

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

plt.style.use('/Users/pxm588@student.bham.ac.uk/PhD/cfa_spectra_pipeline/src/GausSN.mplstyle')

# Define paths to the datasets
datasets = {
    'CfA': {
        'path': '../../output/cfa_spectra_analysis/cfa_found_dataset.csv',
        't_true': 'phase',
        't_snid': 'SNID_age',
        't_err': 'SNID_err',
        'subtype': 'Wang_class'
    },
    'CSP': {
        'path': '../../output/csp_spectra_analysis/csp_found_dataset.csv',
        't_true': 'phase',
        't_snid': 'bootstrap_age',
        't_err': 'bootstrap_err',
        'subtype': 'sub_type'
    },
    # 'Modjaz (Fixed Subtype)': {
    #     'path': '../../output/modjaz_fixed_subtype_spectra_analysis/modjaz_fixed_subtype_found_dataset.csv',
    #     't_true': 'true_age_rest',
    #     't_snid': 'bootstrap_age',
    #     't_err': 'snid_std_dev'
    # },
    # 'Modjaz (Fixed Type)': {
    #     'path': '../../output/modjaz_fixed_type_spectra_analysis/modjaz_fixed_type_found_dataset.csv',
    #     't_true': 'true_age_rest',
    #     't_snid': 'bootstrap_age',
    #     't_err': 'snid_std_dev'
    # },
    'ZTF': {
        'path': '../../output/ztf_spectra_analysis/ztf_found_dataset.csv',
        't_true': 'true_age_rest',
        't_snid': 'bootstrap_age',
        't_err': 'snid_std_dev',
        'subtype': 'sub_type'
    }
}

def load_and_clean(name, config):
    df = pd.read_csv(config['path'])
    # Select relevant columns
    df = df[[config['t_true'], config['t_snid'], config['t_err'], config['subtype']]].dropna()
    df.columns = ['t_true', 't_snid', 't_err', 'subtype']
    # Ensure numeric
    df['t_true'] = pd.to_numeric(df['t_true'], errors='coerce')
    df['t_snid'] = pd.to_numeric(df['t_snid'], errors='coerce')
    df['t_err'] = pd.to_numeric(df['t_err'], errors='coerce')

    df = df.dropna()
    df['residual'] = df['t_snid'] - df['t_true']
    df['norm_residual'] = df['residual'] / df['t_err']
    df['dataset'] = name
    return df

all_data = []
for name, config in datasets.items():
    try:
        all_data.append(load_and_clean(name, config))
        print(f"Loaded {name} with {len(all_data[-1])} points.")
    except Exception as e:
        print(f"Error loading {name}: {e}")

combined_df = pd.concat(all_data, ignore_index=True)
combined_df = combined_df[
    (combined_df['norm_residual'] <= 50)
]

Error loading CfA: [Errno 2] No such file or directory: '../../output/cfa_spectra_analysis/cfa_found_dataset.csv'
Error loading CSP: [Errno 2] No such file or directory: '../../output/csp_spectra_analysis/csp_found_dataset.csv'
Error loading ZTF: [Errno 2] No such file or directory: '../../output/ztf_spectra_analysis/ztf_found_dataset.csv'


ValueError: No objects to concatenate

In [ ]:
print(len(combined_df))
print(combined_df.tail())
print(datasets.keys())

In [ ]:
ztf_early = combined_df[
    (combined_df['dataset'] == 'ZTF') &
    (combined_df['t_true'] >= -15) &
    (combined_df['t_true'] <= -13) &
    (combined_df['t_true'] != -14.629765)
]

print(ztf_early)

In [ ]:
def filter_subtypes(df):
    if df['dataset'].iloc[0] == 'CfA':
        return df[df['subtype'].isin(['N', 'HV'])]
    else:  # CSP and ZTF
        return df[df['subtype'].str.lower() == 'norm']

all_data_filtered = [filter_subtypes(df) for df in all_data]
combined_df = pd.concat(all_data_filtered, ignore_index=True)

# Quick check
for name in datasets.keys():
    subset = combined_df[combined_df['dataset'] == name]
    print(f"{name}: {len(subset)} points | subtypes: {subset['subtype'].unique()}")

In [ ]:
plot_df = combined_df[(combined_df['t_true'] >= -15) & (combined_df['t_true'] <= 25)]

fig, axes = plt.subplots(len(datasets), 1, figsize=(10, 6 * len(datasets)), sharex=True)

for ax, name in zip(axes, datasets.keys()):
    subset = plot_df[plot_df['dataset'] == name]
    ax.errorbar(
        subset['t_true'], subset['residual'],
        yerr=subset['t_err'],
        fmt='o', alpha=0.6, markersize=4, label=name
    )
    ax.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlim(-15, 25)
    ax.set_ylabel('residual (t_snid - t_true) [days]')
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('True Age [days]')
plt.tight_layout()
plt.show()

In [ ]:
# Filter to requested range
plot1_df = combined_df.copy()
plot1_df = plot1_df[(plot1_df['t_true'] >= -15) & (plot1_df['t_true'] <= 25)]

# Binning and calculating metrics
bin_size = 5
bins = np.arange(-15, 26, bin_size)
plot1_df['age_bin'] = pd.cut(plot1_df['t_true'], bins=bins)

def get_metrics(group):
    return pd.Series({
        'bias': group['residual'].mean(),
        'spread': group['residual'].std(),
        'rmse': np.sqrt((group['residual']**2).mean()),
        'count': len(group),
        'mean_err': group['t_err'].mean()
    })

metrics_df = plot1_df.groupby(['dataset', 'age_bin']).apply(get_metrics).reset_index()

# Convert age_bin to midpoint for plotting
metrics_df['bin_midpoint'] = metrics_df['age_bin'].apply(lambda x: x.mid if pd.notnull(x) else np.nan)

os.makedirs('../../output/comparison_analysis', exist_ok=True)
metrics_df.to_csv('../../output/comparison_analysis/comparison_metrics.csv', index=False)
print("Metrics saved to output/comparison_analysis/comparison_metrics.csv")

print(metrics_df.head())

In [ ]:
STYLE_FILE = '../../src/GausSN.mplstyle'
plt.style.use(STYLE_FILE)

In [ ]:
# Plotting Bias
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['bias'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['bias'], marker='o', label=name)

plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('Bias (t_snid - t_true) [days]')
# plt.title('SNID Age Bias vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/bias_comparison.png')
plt.show()

# Plotting Spread
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['spread'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['spread'], marker='s', label=name)

plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('Spread (std of residuals) [days]')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/spread_comparison.png')
plt.show()

# Plotting RMSE
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['rmse'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['rmse'], marker='s', label=name)

plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('rmse of residuals [days]')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/rmse_comparison.png')
plt.show()

# Plotting RMSE
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['mean_err'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['mean_err'], marker='s', label=name)

plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('mean snid error [days]')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/mean_snid_err_comparison.png')
plt.show()

In [ ]:
# Filter to requested range
plot2_df= combined_df.copy()
plot2_df = plot2_df[(plot2_df['t_true'] >= -15) & (plot2_df['t_true'] <= 25)]

# Binning and calculating metrics
bin_size = 2
bins = np.arange(-15, 26, bin_size)
plot2_df['age_bin'] = pd.cut(plot2_df['t_true'], bins=bins)

def get_metrics(group):
    return pd.Series({
        'bias': group['residual'].mean(),
        'spread': group['residual'].std(),
        'rmse': np.sqrt((group['residual']**2).mean()),
        'count': len(group),
        'mean_err': group['t_err'].mean(),
        'chi2_reduced': (group['norm_residual']**2).mean(),
        'mad': group['residual'].abs().median(),
        'coverage_1sigma': (group['residual'].abs() < group['t_err']).mean(),
        'coverage_2sigma': (group['residual'].abs() < 2*group['t_err']).mean()

    })


metrics_df = plot2_df.groupby(['dataset', 'age_bin']).apply(get_metrics).reset_index()

# metrics_df = metrics_df.dropna(subset=['chi2_reduced'])
# metrics_df = metrics_df[
#     (metrics_df['chi2_reduced']<=100)
# ]

# Convert age_bin to midpoint for plotting
metrics_df['bin_midpoint'] = metrics_df['age_bin'].apply(lambda x: x.mid if pd.notnull(x) else np.nan)

# # os.makedirs('../../output/comparison_analysis', exist_ok=True)
metrics_df.to_csv('../../output/comparison_analysis/2daybin_comparison_metrics.csv', index=False)
# print("Metrics saved to output/comparison_analysis/comparison_metrics.csv")

In [ ]:
# Plotting Bias
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['bias'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['bias'], marker='o', label=name)

plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('Bias (t_snid - t_true) [days]')
# plt.title('SNID Age Bias vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
# plt.savefig('../../output/comparison_analysis/bias_comparison.png')
plt.show()

# Plotting Spread
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['spread'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['spread'], marker='s', label=name)

plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('Spread (std of residuals) [days]')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
# plt.savefig('../../output/comparison_analysis/spread_comparison.png')
plt.show()

# Plotting RMSE
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['rmse'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['rmse'], marker='s', label=name)

plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('rmse of residuals [days]')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/rmse_comparison.png')
plt.show()

# Plotting RMSE
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[metrics_df['dataset'] == name].dropna(subset=['mean_err'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['mean_err'], marker='s', label=name)

plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('mean snid error [days]')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/mean_snid_err_comparison.png')
plt.show()

In [ ]:

# Plotting Reduced Chi-Squared
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[(metrics_df['dataset'] == name) & (metrics_df['chi2_reduced'] <= 200)].dropna(subset=['chi2_reduced'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['chi2_reduced'], marker='s', label=name)

plt.axhline(1, color='red', linestyle='--', alpha=0.5, label='Ideal ($\\chi^2_\\nu=1$)')
plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('Reduced $\\chi^2$')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/chi2_reduced_comparison.png')
plt.show()

In [ ]:

# Plotting Reduced MAD
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[(metrics_df['dataset'] == name)].dropna(subset=['mad'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['mad'], marker='s', label=name)

# plt.axhline(1, color='red', linestyle='--', alpha=0.5, label='Ideal ($\\chi^2_\\nu=1$)')
plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('MAD')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/mad_comparison.png')
plt.show()

In [ ]:

# Plotting 1sigma coverage
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[(metrics_df['dataset'] == name)].dropna(subset=['coverage_1sigma'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['coverage_1sigma'], marker='s', label=name)

# plt.axhline(1, color='red', linestyle='--', alpha=0.5, label='Ideal ($\\chi^2_\\nu=1$)')
plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('1-sigma coverage')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/coverage_1sigma_comparison.png')
plt.show()

In [ ]:

# Plotting 2sigma coverage
plt.figure(figsize=(10, 6))
for name in datasets.keys():
    subset = metrics_df[(metrics_df['dataset'] == name)].dropna(subset=['coverage_2sigma'])
    if not subset.empty:
        plt.plot(subset['bin_midpoint'], subset['coverage_2sigma'], marker='s', label=name)

# plt.axhline(1, color='red', linestyle='--', alpha=0.5, label='Ideal ($\\chi^2_\\nu=1$)')
plt.xlim(-15, 25)
plt.xlabel('True Age [days]')
plt.ylabel('2-sigma coverage')
# plt.title('SNID Age Spread vs True Age (-15 to 25 days, 5-day bins)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('../../output/comparison_analysis/coverage_2sigma_comparison.png')
plt.show()